In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC

In [2]:
chemins = [
    Path("data/raw/student_mental_health.csv"),
    Path("../data/raw/student_mental_health.csv"),
    Path("student_mental_health.csv")
]

chemin_csv = next((p for p in chemins if p.exists()), None)
if chemin_csv is None:
    raise FileNotFoundError("Placez student_mental_health.csv dans data/raw/")

df = pd.read_csv(chemin_csv)
print("Dimensions initiales :", df.shape)
display(df.head(3))

Dimensions initiales : (2000, 17)


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium


In [3]:
cible = "Mental_Health_Risk"

print("Cibles manquantes :", df[cible].isna().sum())

df = df.dropna(subset=[cible]).copy()
X = df.drop(columns=["Student_ID", cible])
y = df[cible]

print("Variables explicatives :", X.shape[1])
print("Classes :", sorted(y.unique()))

Cibles manquantes : 0
Variables explicatives : 15
Classes : ['High', 'Low', 'Medium']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("\nClasses dans le train :")
print(y_train.value_counts())
print("\nClasses dans le test :")
print(y_test.value_counts())

X_train : (1600, 15)
X_test  : (400, 15)

Classes dans le train :
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Classes dans le test :
Mental_Health_Risk
High      136
Medium    132
Low       132
Name: count, dtype: int64


In [5]:
colonnes_num = X_train.select_dtypes(include="number").columns.tolist()
colonnes_cat = X_train.select_dtypes(exclude="number").columns.tolist()

print("Colonnes numériques :", colonnes_num)
print("\nColonnes catégorielles :", colonnes_cat)

Colonnes numériques : ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']

Colonnes catégorielles : ['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


In [6]:
print("Valeurs manquantes dans X_train :")
print(X_train.isna().sum().sort_values(ascending=False))

print("\nValeurs manquantes dans X_test :")
print(X_test.isna().sum().sort_values(ascending=False))

Valeurs manquantes dans X_train :
Financial_Stress                  76
Diet_Quality                      55
Sleep_Hours                       54
Social_Media_Hours                48
Attendance_Rate                   39
Family_Support                    36
GPA                               36
Study_Hours_Per_Week               0
Age                                0
Gender                             0
Year_of_Study                      0
Screen_Time_Hours                  0
Physical_Activity                  0
Part_Time_Job                      0
Previous_Mental_Health_History     0
dtype: int64

Valeurs manquantes dans X_test :
Family_Support                    18
Financial_Stress                  15
Diet_Quality                      14
Social_Media_Hours                10
Attendance_Rate                    8
Sleep_Hours                        8
GPA                                2
Study_Hours_Per_Week               0
Age                                0
Gender                         

In [7]:
imputer_num = SimpleImputer(strategy="median")
imputer_cat = SimpleImputer(strategy="most_frequent")

print("Imputation numérique : médiane")
print("Imputation catégorielle : modalité la plus fréquente")

Imputation numérique : médiane
Imputation catégorielle : modalité la plus fréquente


In [8]:
q1 = X_train[colonnes_num].quantile(0.25)
q3 = X_train[colonnes_num].quantile(0.75)
iqr = q3 - q1

borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr

outliers = (
    (X_train[colonnes_num] < borne_basse) |
    (X_train[colonnes_num] > borne_haute)
).sum()

print(outliers.sort_values(ascending=False))

Attendance_Rate         10
Social_Media_Hours      10
Sleep_Hours              9
Study_Hours_Per_Week     7
Screen_Time_Hours        7
GPA                      5
Age                      2
Year_of_Study            0
Financial_Stress         0
Family_Support           0
dtype: int64


In [9]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)
        iqr = q3 - q1
        self.basse_ = q1 - 1.5 * iqr
        self.haute_ = q3 + 1.5 * iqr
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.clip(self.basse_, self.haute_, axis=1).to_numpy()